# RETFound Encoding v2 — EyePACS + Messidor + OLIVES

**Runtime required:** GPU → Runtime → Change runtime type → T4 GPU

**Run cells in order. Start from Cell 0 every fresh session.**

In [ ]:
# ── Cell 1: Load RETFound ──────────────────────────────────────────────────
print('Loading RETFound...')
model = timm.create_model('hf_hub:bitfount/RETFound_MAE', pretrained=True, num_classes=0)
model.eval().to(DEVICE)
for p in model.parameters():
    p.requires_grad = False

TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

with torch.no_grad():
    DIM = model(torch.zeros(1, 3, 224, 224).to(DEVICE)).shape[-1]
print(f'RETFound loaded. Embedding dim: {DIM}')

In [ ]:
# ── Cell 1: Load RETFound ──────────────────────────────────────────────────
print('Loading RETFound...')
model = timm.create_model('hf_hub:bitfound/RETFound_MAE', pretrained=True, num_classes=0)
model.eval().to(DEVICE)
for p in model.parameters():
    p.requires_grad = False

TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

with torch.no_grad():
    DIM = model(torch.zeros(1, 3, 224, 224).to(DEVICE)).shape[-1]
print(f'RETFound loaded. Embedding dim: {DIM}')

In [ ]:
# ── Cell 2: EyePACS — Extract outer zips → inner split parts ───────────────
# Extracts each outer zip one at a time, deletes it immediately to save space.

INNER_DIR = '/content/eyepacs/inner'
os.makedirs(INNER_DIR, exist_ok=True)

for i in range(1, 6):
    outer = f'/content/eyepacs/train.zip.00{i}.zip'
    inner = f'{INNER_DIR}/train.zip.00{i}'
    if os.path.exists(inner):
        print(f'Already extracted: train.zip.00{i}')
        if os.path.exists(outer): os.remove(outer)
        continue
    if not os.path.exists(outer):
        print(f'MISSING: {outer}')
        continue
    print(f'Extracting train.zip.00{i}.zip...')
    with zipfile.ZipFile(outer) as zf:
        zf.extractall(INNER_DIR)
    os.remove(outer)
    _, _, free = shutil.disk_usage('/')
    print(f'  Done. Free: {free//1e9:.1f} GB')

parts = sorted(glob.glob(f'{INNER_DIR}/train.zip.*'))
print(f'\nInner parts: {[os.path.basename(p) for p in parts]}')
_, _, free = shutil.disk_usage('/')
print(f'Free disk: {free//1e9:.1f} GB')

In [ ]:
# ── Cell 3: EyePACS — Stream extract + encode simultaneously ───────────────
# 7-zip extracts images into /content/eyepacs/temp/
# Encoder thread processes and deletes images as fast as they appear.
# Disk never fills up because images are deleted as soon as encoded.

TEMP_IMGS = '/content/eyepacs/temp'
os.makedirs(TEMP_IMGS, exist_ok=True)

labels_df = pd.read_csv('/content/eyepacs/trainLabels.csv')
label_map = dict(zip(labels_df['image'], labels_df['level']))
print(f'Labels: {len(label_map)} entries')

all_embs, all_lbls, processed = [], [], set()
done_event = threading.Event()
lock = threading.Lock()

def encoder_thread():
    BATCH = 64
    while not done_event.is_set() or any(
        f.endswith('.jpeg') for f in os.listdir(TEMP_IMGS)):
        try:
            new = [f for f in os.listdir(TEMP_IMGS)
                   if f.endswith('.jpeg') and f not in processed]
        except:
            time.sleep(0.5); continue
        if not new:
            time.sleep(0.5); continue
        for i in range(0, len(new), BATCH):
            batch = new[i:i+BATCH]
            imgs, lbls = [], []
            for fname in batch:
                path = f'{TEMP_IMGS}/{fname}'
                try:
                    img = Image.open(path).convert('RGB')
                    img.load()
                    imgs.append(TRANSFORM(img))
                    lbls.append(label_map.get(os.path.splitext(fname)[0], -1))
                except: pass
                finally:
                    try: os.remove(path)
                    except: pass
                    with lock: processed.add(fname)
            if imgs:
                with torch.no_grad():
                    e = model(torch.stack(imgs).to(DEVICE)).cpu().numpy()
                with lock:
                    all_embs.append(e)
                    all_lbls.extend(lbls)

t = threading.Thread(target=encoder_thread, daemon=True)
t.start()

print('Starting 7-zip extraction (encoder running in parallel)...')
result = subprocess.run(
    ['7z', 'e', f'{INNER_DIR}/train.zip.001', f'-o{TEMP_IMGS}', '-y', '-bd'],
    capture_output=True, text=True
)
print(f'7-zip finished (exit code {result.returncode})')
if result.returncode != 0:
    print('Error:', result.stderr[-500:])

print('Waiting for encoder to finish...')
time.sleep(5)
done_event.set()
t.join(timeout=120)

if all_embs:
    embs = np.concatenate(all_embs, axis=0)
    lbls = np.array(all_lbls)
    np.save(f'{OUT_DIR}/eyepacs_retfound.npy', embs)
    np.save(f'{OUT_DIR}/eyepacs_labels.npy', lbls)
    print(f'EyePACS saved: embeddings={embs.shape}, labels={lbls.shape}')
else:
    print('ERROR: No embeddings collected.')

shutil.rmtree(INNER_DIR, ignore_errors=True)
shutil.rmtree(TEMP_IMGS, ignore_errors=True)
_, _, free = shutil.disk_usage('/')
print(f'Free after cleanup: {free//1e9:.1f} GB')

In [ ]:
# ── Cell 4: Messidor — Upload IMAGES4.zip and encode ──────────────────────
# Upload IMAGES4.zip from your laptop: data/raw/messidor/IMAGES4.zip (~241 MB)

print('Upload IMAGES4.zip from your laptop (data/raw/messidor/IMAGES4.zip):')
uploaded = files.upload()

MESSIDOR_DIR = '/content/messidor'
os.makedirs(MESSIDOR_DIR, exist_ok=True)
for fname in uploaded:
    print(f'Extracting {fname}...')
    subprocess.run(['7z', 'e', fname, f'-o{MESSIDOR_DIR}', '-y', '-bd'],
                   capture_output=True)
    os.remove(fname)

img_paths = sorted(p for p in Path(MESSIDOR_DIR).rglob('*')
                   if p.suffix.lower() in ['.png', '.jpg', '.jpeg'])
print(f'Messidor images: {len(img_paths)}')

class FolderDataset(Dataset):
    def __init__(self, paths): self.paths = paths
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        return TRANSFORM(Image.open(self.paths[i]).convert('RGB')), 0

@torch.no_grad()
def encode_folder(dataset, batch_size=64):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    embs = [model(imgs.to(DEVICE)).cpu().numpy() for imgs, _ in tqdm(loader)]
    return np.concatenate(embs, axis=0)

embs_m = encode_folder(FolderDataset(img_paths))
np.save(f'{OUT_DIR}/messidor_retfound.npy', embs_m)
print(f'Messidor saved: {embs_m.shape}')
shutil.rmtree(MESSIDOR_DIR, ignore_errors=True)

In [ ]:
# ── Cell 5: OLIVES — Stream from HuggingFace and encode ───────────────────
# No upload needed — streams directly from HuggingFace.

!pip install datasets -q
from datasets import load_dataset
import PIL

print('Streaming OLIVES from HuggingFace...')
ds = load_dataset('gOLIVES/OLIVES_Dataset', 'disease_classification',
                  split='train', streaming=True)

embs_o, eye_ids, lbls_o = [], [], []
buf_imgs, buf_eyes, buf_lbls = [], [], []
BATCH = 32
n = 0

def flush():
    if not buf_imgs: return
    with torch.no_grad():
        embs_o.append(model(torch.stack(buf_imgs).to(DEVICE)).cpu().numpy())
    eye_ids.extend(buf_eyes)
    lbls_o.extend(buf_lbls)
    buf_imgs.clear(); buf_eyes.clear(); buf_lbls.clear()

for sample in ds:
    try:
        img = sample['image']
        if not isinstance(img, PIL.Image.Image):
            img = PIL.Image.fromarray(img)
        buf_imgs.append(TRANSFORM(img.convert('RGB')))
        buf_eyes.append(sample.get('Eye_ID', -1))
        buf_lbls.append(sample.get('Disease Label', -1))
        n += 1
        if len(buf_imgs) >= BATCH: flush()
        if n % 1000 == 0: print(f'  {n} images encoded...')
    except: pass
flush()

embs_o = np.concatenate(embs_o, axis=0)
np.save(f'{OUT_DIR}/olives_retfound.npy', embs_o)
np.save(f'{OUT_DIR}/olives_eye_ids.npy', np.array(eye_ids))
np.save(f'{OUT_DIR}/olives_labels.npy', np.array(lbls_o))
print(f'OLIVES saved: {embs_o.shape}, unique eyes: {len(np.unique(eye_ids))}')

In [ ]:
# ── Cell 6: Validate ──────────────────────────────────────────────────────
print('=== Embedding Validation ===')
for fname in os.listdir(OUT_DIR):
    arr = np.load(f'{OUT_DIR}/{fname}')
    size = os.path.getsize(f'{OUT_DIR}/{fname}')
    print(f'{fname}: shape={arr.shape}, mean={arr.mean():.4f}, std={arr.std():.4f} | {size/1e6:.1f} MB')

In [ ]:
# ── Cell 7: Done ───────────────────────────────────────────────────────────
# Embeddings are saved to Google Drive: My Drive/synapse/embeddings/
# Download them from drive.google.com and place in:
#   data/processed/embeddings/ on your laptop
print('All embeddings in Google Drive:')
for f in os.listdir(OUT_DIR):
    print(f'  /synapse/embeddings/{f}')
print('\nDownload from drive.google.com → put in data/processed/embeddings/')